# Preprocessing pipeline

## Imports and constants

In [1]:
import os
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset
from tqdm.notebook import tqdm
MAX_LEN = 512

CHAR_TO_IDX = {
    'A':0,
    'C':1,
    'G':2,
    'T':3,
    'N':4,
    'P':5
}
MAX_LEN = 512

USECOLS = [
    'sequence',
    'cdr3_start','cdr3_end',
    'v_sequence_start','v_sequence_end',
    'j_sequence_start','j_sequence_end',
    'v_score','v_identity',
    'j_score','j_identity',
    'locus','sequence_id', 'rev_comp'
]

## Config

In [2]:
IG_PATH = "../data/train_5494024.tsv"
TCR_PATH = "../data/train_5494024_tcr.tsv"

OUT_PATH = "../data/train_out_5494024.pt"

NEG_RATIO = 2

## Fastq aggregation

In [3]:
def load_and_aggregate(ig_path, tcr_path):

    print("=" * 60)
    print("STARTING IG/TCR AGGREGATION")
    print("=" * 60)

    CHUNK = 1_000_000

    def process_file(path):

        print(f"Processing file: {path}")

        dfs = []
        total_chunks = 0
        total_rows = 0

        for chunk_id, chunk in enumerate(
            pd.read_csv(
                path,
                usecols=USECOLS,
                delimiter="\t",
                chunksize=CHUNK
            )
        ):

            total_chunks += 1
            total_rows += len(chunk)

            print(
                f"[{os.path.basename(path)}] "
                f"Chunk {chunk_id+1} | rows={len(chunk):,}"
            )

            v = (
                chunk.sort_values("v_identity", ascending=False)
                .drop_duplicates("sequence_id")
                [[
                    "sequence_id",
                    "v_sequence_start",
                    "v_sequence_end",
                    "v_score",
                    "v_identity"
                ]]
            )

            j = (
                chunk.sort_values("j_identity", ascending=False)
                .drop_duplicates("sequence_id")
                [[
                    "sequence_id",
                    "j_sequence_start",
                    "j_sequence_end",
                    "j_score",
                    "j_identity"
                ]]
            )

            cdr3 = (
                chunk.dropna(subset=["cdr3_start"])
                .drop_duplicates("sequence_id")
                [[
                    "sequence_id",
                    "cdr3_start",
                    "cdr3_end"
                ]]
            )

            seq = (
                chunk[["sequence_id","sequence"]]
                .drop_duplicates("sequence_id")
            )

            df = seq.merge(v, how="left", on="sequence_id")
            df = df.merge(j, how="left", on="sequence_id")
            df = df.merge(cdr3, how="left", on="sequence_id")

            dfs.append(df)

        print(
            f"Finished {path} | "
            f"chunks={total_chunks} | "
            f"rows={total_rows:,}"
        )

        result = pd.concat(dfs)

        print(
            f"Aggregated dataframe shape: {result.shape}"
        )

        return result

    ig = process_file(ig_path)
    tcr = process_file(tcr_path)

    print("Computing quality scores...")

    def score(df):
        return (
            df.v_score.fillna(0) +
            df.j_score.fillna(0) +
            2*(df.v_identity.fillna(0) + df.j_identity.fillna(0))
        )

    ig["score"] = score(ig)
    tcr["score"] = score(tcr)

    print("Merging IG/TCR datasets...")

    merged = pd.merge(
        ig,
        tcr,
        on="sequence_id",
        how="outer",
        suffixes=("_ig","_tcr")
    )

    print(f"Merged shape: {merged.shape}")

    choose_ig = (
        merged.score_ig.fillna(-1)
        >=
        merged.score_tcr.fillna(-1)
    )

    final = pd.DataFrame({
        "sequence": np.where(
            choose_ig,
            merged.sequence_ig,
            merged.sequence_tcr
        ),

        "sequence_id": merged.sequence_id,

        "v_sequence_start": np.where(
            choose_ig,
            merged.v_sequence_start_ig,
            merged.v_sequence_start_tcr
        ),

        "v_sequence_end": np.where(
            choose_ig,
            merged.v_sequence_end_ig,
            merged.v_sequence_end_tcr
        ),

        "j_sequence_start": np.where(
            choose_ig,
            merged.j_sequence_start_ig,
            merged.j_sequence_start_tcr
        ),

        "j_sequence_end": np.where(
            choose_ig,
            merged.j_sequence_end_ig,
            merged.j_sequence_end_tcr
        ),

        "cdr3_start": np.where(
            choose_ig,
            merged.cdr3_start_ig,
            merged.cdr3_start_tcr
        ),

        "cdr3_end": np.where(
            choose_ig,
            merged.cdr3_end_ig,
            merged.cdr3_end_tcr
        ),

        "best_v_identity": np.where(
            choose_ig,
            merged.v_identity_ig,
            merged.v_identity_tcr
        ),

        "best_j_identity": np.where(
            choose_ig,
            merged.j_identity_ig,
            merged.j_identity_tcr
        ),

        "best_v_score": np.where(
            choose_ig,
            merged.v_score_ig,
            merged.v_score_tcr
        ),

        "best_j_score": np.where(
            choose_ig,
            merged.j_score_ig,
            merged.j_score_tcr
        ),
    })

    final["has_v"] = final.v_sequence_start.notna()
    final["has_j"] = final.j_sequence_start.notna()

    print("=" * 60)
    print("FINAL DATASET SUMMARY")
    print("=" * 60)
    print(f"Final shape: {final.shape}")
    print(f"Has V: {final.has_v.sum():,}")
    print(f"Has J: {final.has_j.sum():,}")

    return final

In [4]:
class VDJDataset(Dataset):

    def __init__(self, ig_path, tcr_path, neg_ratio=1):

        print("=" * 60)
        print("BUILDING DATASET")
        print("=" * 60)

        df = load_and_aggregate(ig_path, tcr_path)

        print("Applying positive filters...")

        pos_mask = (
            df.has_v &
            df.has_j &
            (df.best_v_identity > 80) &
            (df.best_j_identity > 88) &
            (df.best_v_score > 50) &
            (df.best_j_score > 40)
        )

        pos = df[pos_mask].copy()

        print(f"Positive samples: {len(pos):,}")

        print("Sampling negatives...")

        neg = df[
            (~df.has_v) &
            (~df.has_j)
        ].sample(
            n=min(len(pos)*neg_ratio, len(df)),
            random_state=42
        )

        print(f"Negative samples: {len(neg):,}")

        self.data = pd.concat([pos, neg]).reset_index(drop=True)
        self.pos_len = len(pos)

        print("=" * 60)
        print("DATASET READY")
        print("=" * 60)
        print(f"Total samples: {len(self.data):,}")
        print(f"Positive ratio: {self.pos_len / len(self.data):.3f}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        if idx % 100000 == 0 and idx > 0:
            print(f"Processed {idx:,} samples")

        row = self.data.iloc[idx]

        seq = row.sequence[:MAX_LEN]
        seq = seq + "P"*(MAX_LEN-len(seq))

        X = torch.tensor(
            [CHAR_TO_IDX[c] for c in seq]
        )

        y = torch.zeros((3,MAX_LEN))

        if idx < self.pos_len:
            y[0] = make_vector(
                row.cdr3_start,
                row.cdr3_end
            )

            y[1] = make_vector(
                row.v_sequence_start,
                row.v_sequence_end
            )

            y[2] = make_vector(
                row.j_sequence_start,
                row.j_sequence_end
            )

        return X, y


def make_vector(start, end):

    v = torch.zeros(MAX_LEN)

    if pd.isna(start) or pd.isna(end):
        return v

    v[int(start)-1:int(end)] = 1

    return v


def dataset_to_pt(dataset, out_path):

    print("=" * 60)
    print("SERIALIZING DATASET")
    print("=" * 60)

    X_list = []
    y_list = []

    total = len(dataset)

    for i in range(total):

        if i % 50000 == 0:
            print(
                f"Tensorizing {i:,}/{total:,}"
            )

        x, y = dataset[i]

        X_list.append(x)
        y_list.append(y)

    print("Stacking tensors...")

    X = torch.stack(X_list)
    y = torch.stack(y_list)

    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")

    print(f"Saving dataset -> {out_path}")

    torch.save(
        {'X': X, 'y': y},
        out_path
    )

    print("=" * 60)
    print("DATASET SAVED SUCCESSFULLY")
    print("=" * 60)

## Dataset creation

In [5]:
dataset = VDJDataset(
    ig_path=IG_PATH,
    tcr_path=TCR_PATH,
    neg_ratio=NEG_RATIO
)

print(len(dataset))

BUILDING DATASET
STARTING IG/TCR AGGREGATION
Processing file: ../data/train_5494024.tsv
[train_5494024.tsv] Chunk 1 | rows=1,000,000
[train_5494024.tsv] Chunk 2 | rows=1,000,000
[train_5494024.tsv] Chunk 3 | rows=1,000,000
[train_5494024.tsv] Chunk 4 | rows=1,000,000
[train_5494024.tsv] Chunk 5 | rows=1,000,000
[train_5494024.tsv] Chunk 6 | rows=1,000,000
[train_5494024.tsv] Chunk 7 | rows=1,000,000
[train_5494024.tsv] Chunk 8 | rows=1,000,000
[train_5494024.tsv] Chunk 9 | rows=1,000,000
[train_5494024.tsv] Chunk 10 | rows=1,000,000
[train_5494024.tsv] Chunk 11 | rows=1,000,000
[train_5494024.tsv] Chunk 12 | rows=1,000,000
[train_5494024.tsv] Chunk 13 | rows=1,000,000
[train_5494024.tsv] Chunk 14 | rows=1,000,000
[train_5494024.tsv] Chunk 15 | rows=1,000,000
[train_5494024.tsv] Chunk 16 | rows=1,000,000
[train_5494024.tsv] Chunk 17 | rows=1,000,000
[train_5494024.tsv] Chunk 18 | rows=1,000,000
[train_5494024.tsv] Chunk 19 | rows=1,000,000
[train_5494024.tsv] Chunk 20 | rows=1,000,000
[

## Combining train datasets and saving

In [6]:
dataset_to_pt(dataset, OUT_PATH)

print("Saved:", OUT_PATH)

SERIALIZING DATASET
Tensorizing 0/151,104
Tensorizing 50,000/151,104
Tensorizing 100,000/151,104
Processed 100,000 samples
Tensorizing 150,000/151,104
Stacking tensors...
X shape: torch.Size([151104, 512])
y shape: torch.Size([151104, 3, 512])
Saving dataset -> ../data/train_out_5494024.pt
DATASET SAVED SUCCESSFULLY
Saved: ../data/train_out_5494024.pt


In [8]:
train_18748718, train_5494024=torch.load('../data/train_out_5494024.pt'), torch.load('../data/train_out_18748718.pt')
X_train_18748718, y_train_18748718, X_train_5494024, y_train_5494024 = train_18748718['X'],train_18748718['y'],  train_5494024['X'], train_5494024['y']

/scratch/ipykernel_4091962/3645480668.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_18748718, train_5494024=torch.load('../data/train_out_5494024.pt'), torch.loa

In [9]:
X = torch.Tensor(np.concatenate([X_train_18748718, X_train_5494024], axis=0))
y = torch.Tensor(np.concatenate([y_train_18748718, y_train_5494024], axis =0))

In [10]:
torch.save({'X': X, 'y': y}, '../data/train.pt')